# Public repository note

This notebook is an output-cleared code copy. It requires locally authorised data and is not runnable from the public repository alone. Green Street raw data, intermediate files, derived aggregates and outputs are not distributed.


# 14 RQ3: Retail Adjustment Patterns

This notebook answers the third dissertation research question:

> What distinct post-pandemic retail adjustment patterns can be identified
> across London's office-market MSOAs, and how are these patterns associated
> with commuter-demand shock and office-stock adjustment?

RQ3 is deliberately exploratory. It does not force every office area into a
binary choice between volatility and stagnation. Instead, it identifies
groups of MSOAs with similar combinations of retail-stock, vacancy, turnover
and formation changes. Commuter shock and office restructuring are added only
after the groups have been formed. They therefore help interpret the patterns
without determining them.

The geographical boundary remains the five predefined Central London office
submarkets. Residential-origin retail change is analysed in RQ1 and is not
repeated in RQ3.

In [ ]:
from pathlib import Path
import os
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch
from scipy.stats import kruskal
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

BASE = Path(os.environ.get("DISSERTATION_WORKSPACE", Path.cwd().resolve()))
OUTPUT_DIR = BASE / "outputs" / "restricted_rq3_retail_adjustment_pathways"
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

H2_PANEL_PATH = (
    BASE / "outputs" / "restricted_h2_office_restructuring"
    / "h2_combined_analysis_panel.csv"
)
WORKPLACE_SHOCK_PATH = (
    BASE / "outputs" / "restricted_h1_fine_grained_analysis"
    / "h1_destination_all_submarket_msoa_targets.csv"
)
MSOA_PATH = (
    BASE / "outputs" / "restricted_msoa_origin_exposure_analysis"
    / "london_msoa_2021_boundaries.geojson"
)
OFFICE_MARKETS_PATH = BASE / "London_Office_Markets_V1.geojson"

MIN_BASELINE_STOCK = 20
WINSOR_LOWER = 0.05
WINSOR_UPPER = 0.95
PCA_VARIANCE_TARGET = 0.80
N_PATHWAYS = 5
RANDOM_SEED = 42

PATHWAY_ORDER = [
    "High-turnover expansion",
    "Consumer-facing growth with stock contraction",
    "Low-turnover consolidation",
    "Broad commercial contraction",
    "Vacancy-led stress",
]
PATHWAY_COLORS = {
    "High-turnover expansion": "#d97732",
    "Consumer-facing growth with stock contraction": "#3f806b",
    "Low-turnover consolidation": "#2f8f9d",
    "Broad commercial contraction": "#527aa3",
    "Vacancy-led stress": "#b64f4f",
}
PATTERN_DISPLAY_LABELS = {
    "High-turnover expansion": "High-turnover expansion",
    "Consumer-facing growth with stock contraction": (
        "Consumer-facing growth / stock contraction"
    ),
    "Low-turnover consolidation": "Low-turnover consolidation",
    "Broad commercial contraction": "Broad commercial contraction",
    "Vacancy-led stress": "Vacancy-led stress",
}

sns.set_theme(style="whitegrid", context="talk")

## 1. Construct the workplace retail-change profiles

Each office-market MSOA is represented by its mean 2023-2025 change relative
to 2019. The eight clustering indicators preserve complementary information:

- Green Street recorded active stock, turnover, net formation, vacancy and
  long-term vacancy;
- OpenLocal retail unit stock, floorspace and rateable value.

The OpenLocal occupation-based vacancy proxy is retained for the source
reconciliation and RQ1/RQ2 outcome checks, but is not used as an additional
clustering input. The profiles already include Green Street's consumer-facing
vacancy and long-term-vacancy measures; adding a differently defined
property-occupation rate would give disproportionate weight to non-equivalent
vacancy constructs.

Turnover measures the combined rate of recorded openings and closures, while
net formation distinguishes a market with more openings than closures from
one with the reverse balance. Event dates are Green Street detection dates,
so annual values may lag the underlying business event by up to approximately
six months.

The clustering sample requires at least 20 active Green Street premises and
20 OpenLocal retail units in 2019. For each indicator, values below the fifth
percentile are replaced by the fifth-percentile value and values above the
ninety-fifth percentile are replaced by the ninety-fifth-percentile value.
No MSOA is removed. This limits the influence of a few unusually large
small-area changes. The indicators are then standardised so that measures
recorded as percentages, percentage points and monetary values contribute on
a comparable scale.

In [ ]:
panel = pd.read_csv(H2_PANEL_PATH)

CLUSTER_VARIABLES = [
    "active_retail_log_change_2019",
    "turnover_rate_change_2019",
    "net_formation_rate_change_2019",
    "vacancy_share_change_2019",
    "long_term_vacancy_share_change_2019",
    "ol_retail_unit_log_change",
    "ol_retail_floor_log_change",
    "ol_retail_value_log_change",
]

VARIABLE_LABELS = {
    "active_retail_log_change_2019": "GS active retail stock",
    "turnover_rate_change_2019": "GS turnover rate",
    "net_formation_rate_change_2019": "GS net formation",
    "vacancy_share_change_2019": "GS vacancy",
    "long_term_vacancy_share_change_2019": "GS long-term vacancy",
    "ol_retail_unit_log_change": "OL retail units",
    "ol_retail_floor_log_change": "OL retail floorspace",
    "ol_retail_value_log_change": "OL rateable value",
}

aggregation = {
    **{column: "mean" for column in CLUSTER_VARIABLES},
    "fragmentation_z": "mean",
    "persistent_large_z": "mean",
    "log_active_retail_2019": "first",
    "ol_retail_units_2019": "first",
}
workplace_profiles = (
    panel.groupby(["MSOA21CD", "MSOA21NM", "study_submarket"], as_index=False)
    .agg(aggregation)
)
workplace_profiles["gs_active_2019"] = (
    np.exp(workplace_profiles["log_active_retail_2019"]) - 1
)
workplace_profiles = workplace_profiles[
    workplace_profiles["gs_active_2019"].ge(MIN_BASELINE_STOCK)
    & workplace_profiles["ol_retail_units_2019"].ge(MIN_BASELINE_STOCK)
].dropna(subset=CLUSTER_VARIABLES).reset_index(drop=True)

raw_cluster_values = workplace_profiles[CLUSTER_VARIABLES].copy()
winsorised_values = raw_cluster_values.copy()
winsor_limits = []
for column in CLUSTER_VARIABLES:
    lower = raw_cluster_values[column].quantile(WINSOR_LOWER)
    upper = raw_cluster_values[column].quantile(WINSOR_UPPER)
    winsorised_values[column] = raw_cluster_values[column].clip(lower, upper)
    winsor_limits.append({"variable": column, "lower": lower, "upper": upper})

scaler = StandardScaler()
standardised_values = scaler.fit_transform(winsorised_values)
pca_full = PCA().fit(standardised_values)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
n_components = int(np.argmax(cumulative_variance >= PCA_VARIANCE_TARGET) + 1)
pca = PCA(n_components=n_components)
pca_values = pca.fit_transform(standardised_values)

print("Eligible office-market MSOAs:", len(workplace_profiles))
print("PCA components retained:", n_components)
print("Variance retained:", round(cumulative_variance[n_components - 1], 3))

## 2. Select the number of patterns

Ward hierarchical clustering joins the most similar MSOA profiles while
minimising the increase in within-group variation at each step. Principal
components retaining at least 80% of the original variance are used so that
strongly correlated retail indicators do not receive repeated weight.

The number of groups is not selected from the desired narrative. Solutions
from two to seven groups are compared using:

- the silhouette score, which is higher when observations are more similar to
  their own group than to other groups;
- perturbation stability, measured by the adjusted Rand index after adding
  small random disturbances to the standardised profiles.

Five patterns are retained because this solution has the highest silhouette
score among the tested alternatives and remains stable under small
perturbations. The absolute silhouette score is modest, so the groups should
be interpreted as overlapping empirical patterns rather than sharply
separated market regimes. The labels are assigned only after examining the
resulting profile means.

In [ ]:
selection_rows = []
base_labels_by_k = {}
rng = np.random.default_rng(RANDOM_SEED)

for k in range(2, 8):
    labels = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(
        pca_values
    )
    base_labels_by_k[k] = labels
    stability_values = []
    for _ in range(200):
        perturbed = pca_values + rng.normal(0, 0.05, pca_values.shape)
        perturbed_labels = AgglomerativeClustering(
            n_clusters=k, linkage="ward"
        ).fit_predict(perturbed)
        stability_values.append(adjusted_rand_score(labels, perturbed_labels))
    selection_rows.append({
        "n_clusters": k,
        "silhouette": silhouette_score(pca_values, labels),
        "mean_perturbation_ari": np.mean(stability_values),
        "p10_perturbation_ari": np.quantile(stability_values, 0.10),
        "smallest_cluster": int(np.bincount(labels).min()),
    })

cluster_selection = pd.DataFrame(selection_rows)
display(cluster_selection.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12.8, 5.3))
diagnostic_specs = [
    (
        "silhouette",
        "A. Separation between patterns",
        "Silhouette score",
        "#3f7cac",
        (0.14, 0.22),
    ),
    (
        "mean_perturbation_ari",
        "B. Stability after small input changes",
        "Mean adjusted Rand index",
        "#d97732",
        (0.64, 0.84),
    ),
]
for ax, (column, title, ylabel, colour, ylim) in zip(axes, diagnostic_specs):
    ax.plot(
        cluster_selection["n_clusters"],
        cluster_selection[column],
        marker="o",
        markersize=7,
        linewidth=2.5,
        color=colour,
    )
    selected = cluster_selection[
        cluster_selection["n_clusters"].eq(N_PATHWAYS)
    ].iloc[0]
    ax.scatter(
        [N_PATHWAYS],
        [selected[column]],
        s=130,
        color=colour,
        edgecolor="#22272b",
        linewidth=1.2,
        zorder=4,
    )
    ax.axvline(
        N_PATHWAYS,
        color="#676d75",
        linestyle="--",
        linewidth=1.3,
    )
    ax.annotate(
        f"Five-pattern solution\n{selected[column]:.3f}",
        (N_PATHWAYS, selected[column]),
        xytext=(12, 12),
        textcoords="offset points",
        fontsize=10.5,
        fontweight="bold",
    )
    ax.set_xticks(range(2, 8))
    ax.set_ylim(*ylim)
    ax.set_xlabel("Number of patterns")
    ax.set_ylabel(ylabel)
    ax.set_title(title, loc="left", fontsize=14, fontweight="bold")

fig.suptitle(
    "Selecting the Number of Retail Adjustment Patterns",
    x=0.02,
    ha="left",
    fontsize=20,
    fontweight="bold",
)
fig.tight_layout(rect=[0, 0, 1, 0.91])
fig.savefig(
    FIGURE_DIR / "fig_00_rq3_cluster_selection.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
raw_cluster_labels = base_labels_by_k[N_PATHWAYS]

# Labels are based on the observed standardised profiles produced by this
# fixed data preparation. They describe patterns rather than causal states.
cluster_name_map = {
    0: "Low-turnover consolidation",
    1: "Broad commercial contraction",
    2: "Consumer-facing growth with stock contraction",
    3: "High-turnover expansion",
    4: "Vacancy-led stress",
}
workplace_profiles["raw_cluster"] = raw_cluster_labels
workplace_profiles["pattern"] = workplace_profiles["raw_cluster"].map(
    cluster_name_map
)

standardised_frame = pd.DataFrame(
    standardised_values,
    columns=CLUSTER_VARIABLES,
)
standardised_frame["pattern"] = workplace_profiles["pattern"].values

profile_z = (
    standardised_frame.groupby("pattern")[CLUSTER_VARIABLES]
    .mean()
    .reindex(PATHWAY_ORDER)
)
profile_raw = (
    workplace_profiles.groupby("pattern")[CLUSTER_VARIABLES]
    .mean()
    .reindex(PATHWAY_ORDER)
)
pathway_sizes = (
    workplace_profiles["pattern"].value_counts()
    .reindex(PATHWAY_ORDER)
    .rename("n_msoa")
)

display(pathway_sizes)
display(profile_raw.round(3))

## 3. Figure 1: pattern profiles

The heatmap is the main classification result. Each row is a pattern and each
column is one retail-change indicator. Colours show whether the pattern mean
is above or below the study-sample mean after standardisation. Cell labels
show the actual mean change: percentages for stock, floorspace and value, and
percentage points for turnover, formation and vacancy. The figure must be read
across a row: a pattern is defined by its combination of outcomes, not by one
isolated cell.

In [ ]:
heatmap_data = profile_z.rename(columns=VARIABLE_LABELS)
annotation_data = profile_raw.copy()
log_change_columns = [
    "active_retail_log_change_2019",
    "ol_retail_unit_log_change",
    "ol_retail_floor_log_change",
    "ol_retail_value_log_change",
]
rate_change_columns = [
    "turnover_rate_change_2019",
    "net_formation_rate_change_2019",
    "vacancy_share_change_2019",
    "long_term_vacancy_share_change_2019",
]
for column in log_change_columns:
    annotation_data[column] = annotation_data[column].map(
        lambda value: f"{100 * np.expm1(value):+.1f}%"
    )
for column in rate_change_columns:
    annotation_data[column] = annotation_data[column].map(
        lambda value: f"{100 * value:+.1f} pp"
    )
annotation_data = annotation_data.rename(columns=VARIABLE_LABELS)
short_column_labels = [
    "Active\nstock",
    "Turnover",
    "Net\nformation",
    "Vacancy",
    "Long-term\nvacancy",
    "Retail\nunits",
    "Retail\nfloorspace",
    "Rateable\nvalue",
]
heatmap_data.columns = short_column_labels
annotation_data.columns = short_column_labels
row_labels = [
    f"{PATTERN_DISPLAY_LABELS[pattern]} (n={int(pathway_sizes[pattern])})"
    for pattern in PATHWAY_ORDER
]

fig, ax = plt.subplots(figsize=(13.5, 6.4))
sns.heatmap(
    heatmap_data,
    ax=ax,
    cmap="RdBu_r",
    center=0,
    vmin=-1.6,
    vmax=1.6,
    annot=annotation_data,
    fmt="",
    linewidths=1.0,
    linecolor="white",
    annot_kws={"fontsize": 14},
    cbar_kws={
        "label": "Relative to the study mean (standardised)",
        "shrink": 0.88,
        "aspect": 22,
    },
)
colour_bar = ax.collections[0].colorbar
colour_bar.ax.tick_params(labelsize=13)
colour_bar.set_label(
    "Relative to the study mean (standardised)",
    fontsize=15,
    labelpad=12,
)
ax.set_yticklabels(row_labels, rotation=0, fontsize=15)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, ha="center", fontsize=13)
ax.set_xlabel("")
ax.set_ylabel("")
fig.suptitle(
    "Retail Adjustment Pattern Profiles",
    fontsize=23,
    fontweight="bold",
    x=0.06,
    y=0.99,
    ha="left",
)
ax.text(
    0.3125,
    1.025,
    "Green Street",
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=16,
    fontweight="bold",
)
ax.text(
    0.8125,
    1.025,
    "OpenLocal",
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=16,
    fontweight="bold",
)
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig(
    FIGURE_DIR / "fig_01_rq3_pattern_profile_heatmap.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 4. Figure 2: spatial distribution

The pattern map shows whether similar adjustment profiles are concentrated in
particular parts of the five office submarkets. The patterns remain analytical
types, not official market classifications.

In [ ]:
msoa = gpd.read_file(MSOA_PATH).to_crs("EPSG:27700")
pathway_map = msoa.merge(
    workplace_profiles[
        ["MSOA21CD", "pattern", "study_submarket"]
    ],
    on="MSOA21CD",
    how="inner",
)
office_markets = gpd.read_file(OFFICE_MARKETS_PATH).to_crs("EPSG:27700")
submarket_crosswalk = {
    "Mayfair": "West End", "Soho": "West End", "St James's": "West End",
    "Covent Garden": "West End", "Fitzrovia": "West End",
    "North of Oxford Street": "West End", "Paddington": "West End",
    "Knightsbridge": "West End", "Victoria": "West End", "City Core": "City",
    "Midtown": "Tech Belt & Midtown", "Bloomsbury": "Tech Belt & Midtown",
    "Clerkenwell": "Tech Belt & Midtown", "Euston": "Tech Belt & Midtown",
    "Kings Cross": "Tech Belt & Midtown", "Shoreditch": "Tech Belt & Midtown",
    "Camden": "Tech Belt & Midtown", "Aldgate & Whitechapel": "Tech Belt & Midtown",
    "Canary Wharf": "Canary Wharf", "Southbank": "Southbank",
    "Waterloo": "Southbank", "Vauxhall, Nine Elms and Battersea": "Southbank",
}
office_markets["study_submarket"] = office_markets["Market"].map(submarket_crosswalk)
core_bounds = office_markets.dropna(subset=["study_submarket"]).copy()
core_submarket_boundaries = core_bounds.dissolve(by="study_submarket", as_index=False)
cxmin, cymin, cxmax, cymax = core_bounds.total_bounds

fig, ax = plt.subplots(figsize=(12.4, 8.4))
for pattern in PATHWAY_ORDER:
    pathway_map[pathway_map["pattern"].eq(pattern)].plot(
        ax=ax,
        color=PATHWAY_COLORS[pattern],
        edgecolor="white",
        linewidth=0.42,
    )
core_submarket_boundaries.boundary.plot(ax=ax, color="#30363d", linewidth=1.35, zorder=5)
ax.set_xlim(cxmin - 2500, cxmax + 2500)
ax.set_ylim(cymin - 2500, cymax + 2500)
ax.set_axis_off()
ax.set_title(
    "Retail Adjustment Patterns across the Five Office Submarkets",
    loc="left",
    fontsize=20,
    fontweight="bold",
    pad=12,
)
legend_handles = [
    Patch(
        facecolor=PATHWAY_COLORS[pattern],
        edgecolor="white",
        label=(
            f"{PATTERN_DISPLAY_LABELS[pattern]} "
            f"(n={int(pathway_sizes[pattern])})"
        ),
    )
    for pattern in PATHWAY_ORDER
]
ax.legend(
    handles=legend_handles,
    loc="lower left",
    frameon=True,
    fontsize=14,
    title="Empirically identified pattern",
    title_fontsize=14,
)
fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "fig_02_rq3_pattern_map.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 5. Figure 3: annual consistency check

The clustering uses the mean 2023-2025 profile. The annual lines check whether
the interpretation of each pattern is broadly visible in all three years or
is driven by one unusual year. They are a temporal consistency check rather
than an additional hypothesis test.

In [ ]:
trajectory_panel = panel.merge(
    workplace_profiles[["MSOA21CD", "pattern"]],
    on="MSOA21CD",
    how="inner",
)
trajectory_specs = [
    (
        "active_retail_log_change_2019",
        "Recorded active retail stock",
        lambda x: 100 * (np.exp(x) - 1),
        "Change from 2019 (%)",
    ),
    (
        "turnover_rate_change_2019",
        "Recorded turnover rate",
        lambda x: 100 * x,
        "Change from 2019 (percentage points)",
    ),
    (
        "vacancy_share_change_2019",
        "Recorded vacancy",
        lambda x: 100 * x,
        "Change from 2019 (percentage points)",
    ),
    (
        "ol_retail_unit_log_change",
        "OpenLocal retail units",
        lambda x: 100 * (np.exp(x) - 1),
        "Change from 2019 (%)",
    ),
]

fig, axes = plt.subplots(2, 2, figsize=(10.0, 8.5), sharex=True)
for ax, (column, title, transform, ylabel) in zip(axes.flat, trajectory_specs):
    annual = (
        trajectory_panel.groupby(["pattern", "year"], as_index=False)[column]
        .mean()
    )
    annual["plot_value"] = transform(annual[column])
    for pattern in PATHWAY_ORDER:
        plot = annual[annual["pattern"].eq(pattern)]
        ax.plot(
            plot["year"],
            plot["plot_value"],
            color=PATHWAY_COLORS[pattern],
            linewidth=2.4,
            marker="o",
            markersize=5.5,
            label=PATTERN_DISPLAY_LABELS[pattern],
        )
    ax.axhline(0, color="#777d85", linewidth=1.0)
    ax.set_title(title, loc="left", fontsize=14, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_xticks([2023, 2024, 2025])
    ax.tick_params(labelsize=11)
    ax.grid(axis="y", color="#dfe3e6", linewidth=0.8)
    ax.grid(axis="x", visible=False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=2,
    frameon=False,
    fontsize=12,
    bbox_to_anchor=(0.5, 0.005),
)
fig.suptitle(
    "Annual Consistency of the Retail Adjustment Patterns",
    x=0.02,
    ha="left",
    fontsize=19,
    fontweight="bold",
)
fig.tight_layout(rect=[0, 0.12, 1, 0.93])
fig.savefig(
    FIGURE_DIR / "fig_03_rq3_pattern_trajectories.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 6. Figure 4: links to RQ1 and RQ2

The patterns were formed without commuter or office-structure variables.
These variables are now compared across the five groups:

- the station-based commuter-shock score from RQ1, available only for
  station-containing office MSOAs;
- the office-fragmentation and persistent-large-office measures from RQ2.

Kruskal-Wallis tests are used because pattern sizes are unequal and the
context variables are not assumed to be normally distributed. For the two
RQ2 office-condition comparisons, the final dissertation reports a 9,999-
iteration label-permutation p value alongside the Kruskal-Wallis statistic
and epsilon squared. A significant test indicates that at least one pattern
has a different distribution, but does not establish causality.

In [ ]:
shock = pd.read_csv(WORKPLACE_SHOCK_PATH)[
    ["MSOA21CD", "workplace_msoa_station_shock", "high_affected_group"]
].drop_duplicates("MSOA21CD")
workplace_profiles = workplace_profiles.merge(shock, on="MSOA21CD", how="left")

context_specs = [
    ("workplace_msoa_station_shock", "RQ1 commuter shock", "n=35 station-linked MSOAs"),
    ("fragmentation_z", "RQ2 office fragmentation", "n=81 eligible MSOAs"),
    ("persistent_large_z", "RQ2 persistent large-office structure", "n=81 eligible MSOAs"),
]
def label_permutation_p(groups, observed_h, iterations=9_999, seed=2026):
    """Test whether the observed group separation exceeds random relabelling."""
    values = np.concatenate(groups)
    group_sizes = [len(group) for group in groups]
    boundaries = np.cumsum(group_sizes)[:-1]
    rng = np.random.default_rng(seed)
    simulated_h = np.empty(iterations)
    for iteration in range(iterations):
        shuffled = rng.permutation(values)
        simulated_h[iteration] = kruskal(*np.split(shuffled, boundaries)).statistic
    return (np.sum(simulated_h >= observed_h) + 1) / (iterations + 1)

context_test_rows = []
for column, label, sample_note in context_specs:
    groups = [
        group[column].dropna().to_numpy()
        for _, group in workplace_profiles.groupby("pattern")
        if group[column].notna().sum() >= 2
    ]
    statistic, asymptotic_p = kruskal(*groups)
    label_permutation_p_value = (
        label_permutation_p(groups, statistic)
        if column in {"fragmentation_z", "persistent_large_z"}
        else np.nan
    )
    n_values = len(np.concatenate(groups))
    epsilon_squared = (statistic - len(groups) + 1) / (n_values - len(groups))
    context_test_rows.append({
        "measure": label,
        "kruskal_h": statistic,
        "asymptotic_p": asymptotic_p,
        "label_permutation_p_9999": label_permutation_p_value,
        "epsilon_squared": epsilon_squared,
        "n_observations": workplace_profiles[column].notna().sum(),
        "sample": sample_note,
    })
context_tests = pd.DataFrame(context_test_rows)
display(context_tests.round(3))

fig, axes = plt.subplots(3, 1, figsize=(9.2, 10.5))
for ax, (column, label, sample_note) in zip(axes, context_specs):
    plot = workplace_profiles.dropna(subset=[column]).copy()
    sns.boxplot(
        data=plot,
        x=column,
        y="pattern",
        order=PATHWAY_ORDER,
        palette=PATHWAY_COLORS,
        showfliers=False,
        linewidth=1.1,
        ax=ax,
    )
    sns.stripplot(
        data=plot,
        x=column,
        y="pattern",
        order=PATHWAY_ORDER,
        color="#263238",
        size=3.5,
        alpha=0.62,
        ax=ax,
    )
    result = context_tests.loc[context_tests["measure"].eq(label)].iloc[0]
    test_label = (
        f"H={result['kruskal_h']:.2f}; label-permutation p={result['label_permutation_p_9999']:.3f}; "
        f"epsilon squared={result['epsilon_squared']:.3f}"
        if pd.notna(result['label_permutation_p_9999'])
        else f"Kruskal-Wallis H={result['kruskal_h']:.2f}; p={result['asymptotic_p']:.3f}"
    )
    ax.set_title(
        f"{label}: {test_label}",
        fontsize=14,
        loc="left",
        fontweight="bold",
    )
    ax.set_xlabel(sample_note, fontsize=12)
    ax.set_ylabel("")
    ax.tick_params(axis="y", labelsize=11)
    ax.tick_params(axis="x", labelsize=11)

fig.suptitle(
    "RQ3 Patterns Compared with RQ1 and RQ2 Measures",
    x=0.02,
    ha="left",
    fontsize=19,
    fontweight="bold",
)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(
    FIGURE_DIR / "fig_04_rq3_context_comparison.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 7. Export analysis-ready outputs

The pattern membership file contains confidential-data-derived indicators and
must remain in the restricted output directory. A public repository should
contain processing code, field descriptions and an access statement rather
than these row-level derived values.

In [ ]:
exports = {
    "rq3_pattern_membership.csv": workplace_profiles,
    "rq3_pattern_standardised_profiles.csv": profile_z.reset_index(),
    "rq3_pattern_raw_profiles.csv": profile_raw.reset_index(),
    "rq3_cluster_selection_diagnostics.csv": cluster_selection,
    "rq3_context_tests.csv": context_tests,
}
for filename, dataframe in exports.items():
    dataframe.to_csv(OUTPUT_DIR / filename, index=False)

figure_manifest = pd.DataFrame([
    {
        "figure": "Methodology figure",
        "file": "fig_00_rq3_cluster_selection.png",
        "role": "Selection of the five-pattern solution",
        "placement": "RQ3 methodology",
    },
    {
        "figure": "Figure 1",
        "file": "fig_01_rq3_pattern_profile_heatmap.png",
        "role": "Main pattern definition",
        "placement": "RQ3 results",
    },
    {
        "figure": "Figure 2",
        "file": "fig_02_rq3_pattern_map.png",
        "role": "Spatial distribution",
        "placement": "RQ3 results",
    },
    {
        "figure": "Figure 3",
        "file": "fig_03_rq3_pattern_trajectories.png",
        "role": "Annual consistency check",
        "placement": "RQ3 results",
    },
    {
        "figure": "Figure 4",
        "file": "fig_04_rq3_context_comparison.png",
        "role": "Links to RQ1 and RQ2",
        "placement": "RQ3 results",
    },
])
figure_manifest.to_csv(OUTPUT_DIR / "figure_manifest.csv", index=False)

print("Saved outputs to", OUTPUT_DIR)
display(figure_manifest)